### Check GPU


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU available")

### Upload dataset and project ZIP files


In [ ]:
from google.colab import files
uploaded = files.upload()

### Extract dataset


In [ ]:
import zipfile
import os

zip_path = "/content/BoneFractureYolo8.zip.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content")

dataset_path = "/content/BoneFractureYolo8"

print("Dataset extracted:", os.path.exists(dataset_path))
print("Folders:", os.listdir(dataset_path))

### Extract project


In [ ]:
import zipfile
import os

zip_path = "/content/fracture_project.zip"
extract_path = "/content/fracture_project"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("Project extracted successfully!")
print(os.listdir(extract_path))

### Install Ultralytics


In [ ]:
!pip install ultralytics

### Prepare dataset


In [ ]:
%cd /content/fracture_project

!python training/convert_annotations.py \
    --src /content/BoneFractureYolo8 \
    --out /content/fracture_project/processed_dataset \
    --unlabeled-as-negative

### Validate dataset


In [ ]:
!python training/validate_dataset.py \
    --root /content/fracture_project/processed_dataset

### Train YOLOv8n


In [ ]:
%cd /content/fracture_project

!python training/train.py \
    --data /content/fracture_project/processed_dataset/data.yaml \
    --project /content/fracture_project/runs \
    --epochs 20 \
    --batch 8 \
    --imgsz 640 \
    --export /content/fracture_project/models/best.pt

### Check saved model


In [ ]:
import os

model_path = "/content/fracture_project/models/best.pt"

print("Model exists:", os.path.exists(model_path))
if os.path.exists(model_path):
    print("Model size:", round(os.path.getsize(model_path) / (1024 * 1024), 2), "MB")

### Load model and check classes


In [ ]:
from ultralytics import YOLO

model = YOLO(model_path)

print("Model loaded successfully!")
print("Classes:", model.names)

### Evaluate on test set


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/fracture_project/models/best.pt")

metrics = model.val(
    data="/content/fracture_project/processed_dataset/data.yaml",
    split="test",
    imgsz=640,
    batch=8,
    conf=0.001,
    iou=0.6,
    device=0,
    plots=True
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

### View training results


In [ ]:
from IPython.display import display, Image
import os

run_dir = "/content/fracture_project/runs/fracture_yolov8n"

print(os.listdir(run_dir))

for filename in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    path = os.path.join(run_dir, filename)
    if os.path.exists(path):
        print(filename)
        display(Image(filename=path))

### Download best.pt


In [ ]:
from google.colab import files

files.download(
    "/content/fracture_project/models/best.pt"
)